# Workshop: Building a Retrieval-Augmented Generator (RAG) with LangChain and Pinecone

## Parte 1: RAG Básico con LangChain y OpenAI (FAISS)
En esta primera sección, implementaremos un flujo RAG local utilizando:
- **Carga de documentos:** Lectura de nuestra base de conocimiento local (`data/knowledge.txt`).
- **Text Splitting:** División del texto en fragmentos (chunks) manejables.
- **Embeddings y Vector Store local:** Conversión de texto a vectores usando OpenAI y almacenamiento temporal (FAISS).
- **Retrieval & Generation:** Búsqueda de contexto relevante y generación de respuestas con el modelo de lenguaje de OpenAI.

In [4]:
import os
from dotenv import load_dotenv

# Cargar las variables del archivo .env
load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")
pinecone_key = os.getenv("PINECONE_API_KEY")

if openai_key:
    print("¡OpenAI API Key cargada con éxito!")
else:
    print("Error: No se encontró la OPENAI_API_KEY en el archivo .env")

if pinecone_key:
    print("¡Pinecone API Key cargada con éxito!")
else:
    print("Error: No se encontró la PINECONE_API_KEY en el archivo .env")

¡OpenAI API Key cargada con éxito!
¡Pinecone API Key cargada con éxito!


In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Cargar y dividir el documento local
loader = TextLoader("data/knowledge.txt", encoding="utf-8")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

# 2. Inicializar embeddings y base de datos vectorial local (FAISS)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 3. Configurar el modelo de lenguaje y el prompt
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt_template = """Responde a la pregunta basándote exclusivamente en el siguiente contexto:

{context}

Pregunta: {question}"""
prompt = ChatPromptTemplate.from_template(prompt_template)

def format_docs(documents):
    return "\n\n".join(doc.page_content for doc in documents)

# 4. Construir y ejecutar la cadena RAG local
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query = "¿Qué es Pinecone?"
response = rag_chain.invoke(query)

print(f"Pregunta: {query}")
print(f"\nRespuesta del RAG local (FAISS):\n{response}")

Pregunta: ¿Qué es Pinecone?

Respuesta del RAG local (FAISS):
Pinecone es una base de datos vectorial en la nube diseñada para búsquedas de similitud a gran escala y alta velocidad.


---

## 📐 Arquitectura del Sistema RAG (FAISS vs. Pinecone)

El flujo del sistema se divide en dos enfoques de almacenamiento vectorial para comparar el rendimiento local frente al escalable en la nube:

### 1. Diagrama de Arquitectura General
[ Documento (knowledge.txt) ]
│
▼
[ Text Splitting (Chunks) ]
│
▼
[ OpenAI Embeddings (text-embedding-3-small) ]
│
├─────────────────────────┬─────────────────────────┐
│ (Parte 1: Local)        │ (Parte 2: Cloud)        │
▼                         ▼                         ▼
[ FAISS (RAM) ]         [ Pinecone Serverless ]   <-- (Alta Escalabilidad)
│                         │
└─────────────────────────┴─────────────────────────┘
│
▼
[ Retriever (Top-k Similitud) ]
│
▼
[ Prompt Template + LLM (gpt-4o-mini) ]
│
▼
[ Respuesta Generada al Usuario ]


### 2. Integración de Pinecone con LangChain
La integración de Pinecone con LangChain se realiza a través de la librería `langchain-pinecone` y el componente `PineconeVectorStore`, el cual actúa como un puente directo entre el orquestador de cadenas y la base de datos en la nube:
- **Vectorización y Carga:** Toma los fragmentos de texto procesados (`docs`), genera sus vectores utilizando los embeddings de OpenAI y los almacena de forma persistente en un índice serverless de Pinecone.
- **Recuperación Eficiente (`Retriever`):** Permite transformar el índice en un `AsRetriever`, el cual intercepta las preguntas del usuario, calcula la similitud de coseno a gran velocidad contra los vectores en la nube y extrae únicamente los fragmentos de contexto más precisos (`k=2`).
- **Rol en el RAG:** Elimina las limitaciones de memoria RAM local de FAISS[cite: 8], permitiendo manejar millones de registros con baja latencia y facilitando la persistencia de los datos a nivel empresarial.

## Parte 2: Enhancing the RAG with Pinecone Vector Database
En esta segunda sección, conectamos nuestro RAG a **Pinecone**, una base de datos vectorial en la nube diseñada para almacenar e indexar embeddings a gran escala.

In [6]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Inicializar cliente de Pinecone
load_dotenv()
pinecone_api_key = os.getenv("PINECONE_API_KEY")
if not pinecone_api_key:
    raise ValueError("¡La API Key de Pinecone no está configurada!")

pc = Pinecone(api_key=pinecone_api_key)
index_name = "rag-workshop"

if index_name not in [index.name for index in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Índice '{index_name}' creado exitosamente en Pinecone.")
else:
    print(f"Conectado al índice existente '{index_name}' en Pinecone.")

# 2. Cargar documento y embeddings para Pinecone
loader = TextLoader("data/knowledge.txt", encoding="utf-8")
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

pinecone_vectorstore = PineconeVectorStore.from_documents(
    documents=docs,
    embedding=embeddings,
    index_name=index_name
)
pinecone_retriever = pinecone_vectorstore.as_retriever(search_kwargs={"k": 2})

# 3. Configurar cadena RAG en la nube
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt_template = """Responde a la pregunta basándote exclusivamente en el siguiente contexto:

{context}

Pregunta: {question}"""
prompt = ChatPromptTemplate.from_template(prompt_template)

def format_docs(documents):
    return "\n\n".join(doc.page_content for doc in documents)

pinecone_rag_chain = (
    {"context": pinecone_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query = "¿Qué ventajas ofrece una base de datos vectorial como Pinecone?"
response = pinecone_rag_chain.invoke(query)

print(f"\nPregunta: {query}")
print(f"\nRespuesta del RAG con Pinecone:\n{response}")

Conectado al índice existente 'rag-workshop' en Pinecone.

Pregunta: ¿Qué ventajas ofrece una base de datos vectorial como Pinecone?

Respuesta del RAG con Pinecone:
Una base de datos vectorial como Pinecone ofrece ventajas como búsquedas de similitud a gran escala y alta velocidad, lo que permite manejar grandes volúmenes de datos de manera eficiente y rápida. Esto es especialmente útil para aplicaciones que requieren encontrar relaciones o similitudes entre datos de forma efectiva.
